In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/data_initial_audit.csv")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (50000, 16)


,customer_id,starting_date,suspension_date,observation_end_date,duration_days,event,age,income,annual_premium,has_children,marital_status,home_owner,college_degree,good_credit,county,duration_years
0,2.213007e+11,2022-09-02,2022-09-18,2022-09-18,16,1,42.806,250000.0,756.722144,1.0,Married,1.0,0.0,1.0,Tarrant,0.043806
1,2.213033e+11,2009-09-13,NaN,2022-12-02,4828,0,47.305,125000.0,875.814547,0.0,Married,1.0,1.0,1.0,Tarrant,13.218344
2,2.213023e+11,2018-02-15,NaN,2022-12-02,1751,0,41.892,70000.0,654.552197,1.0,Single,1.0,1.0,1.0,Collin,4.793977
3,2.213030e+11,2016-10-24,NaN,2022-12-02,2230,0,58.639,42500.0,795.654884,0.0,Single,1.0,0.0,1.0,Tarrant,6.105407
4,2.213006e+11,2022-06-24,2022-11-30,2022-11-30,159,1,55.444,125000.0,1142.303520,0.0,Unknown,0.0,0.0,1.0,Denton,0.435318


In [2]:
survival_columns = [
    "duration_days",
    "duration_years",
    "event"
]

display(df[survival_columns].describe())

print("Event values:")
print(sorted(df["event"].dropna().unique()))

print("Negative duration days:",
      (df["duration_days"] < 0).sum())

print("Zero duration days:",
      (df["duration_days"] == 0).sum())

print("Missing survival values:")
display(df[survival_columns].isnull().sum())

,duration_days,duration_years,event
count,50000.000000,50000.000000,50000.00000
mean,3593.909500,9.839588,0.08368
std,2262.827477,6.195284,0.27691
min,1.000000,0.002738,0.00000
25%,1493.000000,4.087611,0.00000
50%,3614.000000,9.894593,0.00000
75%,6291.000000,17.223819,0.00000
max,6291.000000,17.223819,1.00000


Event values:
[np.int64(0), np.int64(1)]
Negative duration days: 0
Zero duration days: 0
Missing survival values:


duration_days     0
duration_years    0
event             0
dtype: int64

In [3]:
survival_df = df[
    [
        "customer_id",
        "duration_days",
        "duration_years",
        "event",
        "annual_premium"
    ]
].copy()

print("Survival dataset shape:", survival_df.shape)
display(survival_df.head())

Survival dataset shape: (50000, 5)


,customer_id,duration_days,duration_years,event,annual_premium
0,2.213007e+11,16,0.043806,1,756.722144
1,2.213033e+11,4828,13.218344,0,875.814547
2,2.213023e+11,1751,4.793977,0,654.552197
3,2.213030e+11,2230,6.105407,0,795.654884
4,2.213006e+11,159,0.435318,1,1142.303520


In [4]:
output_path = "../data/processed/survival_dataset.csv"

survival_df.to_csv(output_path, index=False)

print("Saved:", output_path)

Saved: ../data/processed/survival_dataset.csv


In [8]:
import pandas as pd
import numpy as np

from lifelines import CoxPHFitter

# Load the survival dataset
survival_df = pd.read_csv("../data/processed/survival_dataset.csv")

print("Survival dataset shape:", survival_df.shape)
display(survival_df.head())

# Keep only the columns required for the Cox model
cox_df = survival_df[
    [
        "duration_years",
        "event",
        "annual_premium"
    ]
].copy()

print("Cox dataset shape:", cox_df.shape)
display(cox_df.head())

Survival dataset shape: (50000, 5)


,customer_id,duration_days,duration_years,event,annual_premium
0,2.213007e+11,16,0.043806,1,756.722144
1,2.213033e+11,4828,13.218344,0,875.814547
2,2.213023e+11,1751,4.793977,0,654.552197
3,2.213030e+11,2230,6.105407,0,795.654884
4,2.213006e+11,159,0.435318,1,1142.303520


Cox dataset shape: (50000, 3)


,duration_years,event,annual_premium
0,0.043806,1,756.722144
1,13.218344,0,875.814547
2,4.793977,0,654.552197
3,6.105407,0,795.654884
4,0.435318,1,1142.303520


In [10]:
# Check missing values
print("Missing values before handling:")
display(cox_df.isnull().sum())

# Do not impute duration or event.
# Remove rows with missing survival information.
cox_df = cox_df.dropna(
    subset=["duration_years", "event"]
)

# Impute missing annual premium using the median
cox_df["annual_premium"] = cox_df["annual_premium"].fillna(
    cox_df["annual_premium"].median()
)

print("\nMissing values after handling:")
display(cox_df.isnull().sum())

print("\nFinal Cox dataset shape:", cox_df.shape)

Missing values before handling:


duration_days     0
duration_years    0
event             0
annual_premium    0
dtype: int64


Missing values after handling:


duration_days     0
duration_years    0
event             0
annual_premium    0
dtype: int64


Final Cox dataset shape: (50000, 4)


In [11]:
# Check duration and event values
print("Minimum duration:", cox_df["duration_years"].min())
print("Maximum duration:", cox_df["duration_years"].max())

print("\nEvent values:")
print(cox_df["event"].unique())

print("\nEvent distribution:")
print(cox_df["event"].value_counts())

# Validate duration
assert (cox_df["duration_years"] > 0).all(), (
    "Duration must be greater than zero."
)

# Validate event
assert cox_df["event"].isin([0, 1]).all(), (
    "Event must contain only 0 and 1."
)

print("\nSurvival data validation passed.")

Minimum duration: 0.0027378507871321
Maximum duration: 17.22381930184805

Event values:
[1 0]

Event distribution:
event
0    45816
1     4184
Name: count, dtype: int64

Survival data validation passed.


In [12]:
# Create the Cox model
cox_model = CoxPHFitter(
    penalizer=0.1
)

# Fit the model
cox_model.fit(
    cox_df,
    duration_col="duration_years",
    event_col="event"
)

print("Cox Proportional Hazards model trained successfully.")

Cox Proportional Hazards model trained successfully.


In [13]:
display(cox_model.summary)

,coef,exp(coef),se(coef),coef lower 95%,coef upper 95%,exp(coef) lower 95%,exp(coef) upper 95%,cmp to,z,p,-log2(p)
covariate,,,,,,,,,,,
duration_days,-0.000264,0.999736,0.000005,-0.000275,-0.000254,0.999725,0.999746,0.0,-48.718504,0.000000,inf
annual_premium,0.000020,1.000020,0.000043,-0.000063,0.000104,0.999937,1.000104,0.0,0.477680,0.632878,0.660001


In [14]:
# Evaluate model using the concordance index
concordance_index = cox_model.concordance_index_

print(
    "Cox Concordance Index:",
    round(concordance_index, 4)
)

Cox Concordance Index: 0.9936
